In [1]:
# 1 = Human (X)
# -1 = AI (O)
# 0 = Empty

In [2]:
class TicTacToe:
    def __init__(self):
        self.reset()

    def reset(self):
        self.board = [0] * 9
        self.current_winner = None

    def available_moves(self):
        return [i for i, x in enumerate(self.board) if x == 0]

    def make_move(self, square, player):
        if self.board[square] == 0:
            self.board[square] = player
            if self.check_winner(square, player):
                self.current_winner = player
            return True
        return False

    def check_winner(self, square, player):
        # Row check
        row_ind = square // 3
        row = self.board[row_ind*3:(row_ind+1)*3]
        if all([s == player for s in row]):
            return True

        # Column check
        col_ind = square % 3
        column = [self.board[col_ind+i*3] for i in range(3)]
        if all([s == player for s in column]):
            return True

        # Diagonal check
        if square % 2 == 0:
            diagonal1 = [self.board[i] for i in [0,4,8]]
            if all([s == player for s in diagonal1]):
                return True

            diagonal2 = [self.board[i] for i in [2,4,6]]
            if all([s == player for s in diagonal2]):
                return True

        return False

    def is_draw(self):
        return 0 not in self.board and self.current_winner is None

    def print_board(self):
        symbols = {1: "X", -1: "O", 0: " "}
        for row in [self.board[i*3:(i+1)*3] for i in range(3)]:
            print("| " + " | ".join(symbols[x] for x in row) + " |")


In [3]:
game = TicTacToe()
game.print_board()

game.make_move(0, 1)
game.make_move(4, -1)
game.make_move(1, 1)
game.make_move(8, -1)
game.make_move(2, 1)  # X should win

game.print_board()

print("Winner:", game.current_winner)


|   |   |   |
|   |   |   |
|   |   |   |
| X | X | X |
|   | O |   |
|   |   | O |
Winner: 1


In [4]:
import random
import numpy as np

class QLearningAgent:
    def __init__(self, alpha=0.1, gamma=0.9, epsilon=1.0, epsilon_decay=0.9995, epsilon_min=0.01):

        self.q_table = {}  # (state, action) -> Q value

        self.alpha = alpha
        self.gamma = gamma

        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min

    def get_state(self, board):
        return tuple(board)

    def get_q_value(self, state, action):
        return self.q_table.get((state, action), 0.0)

    def choose_action(self, state, available_moves):
        # Exploration
        if random.random() < self.epsilon:
            return random.choice(available_moves)

        # Exploitation
        q_values = [self.get_q_value(state, a) for a in available_moves]
        max_q = max(q_values)
        best_actions = [a for a, q in zip(available_moves, q_values) if q == max_q]
        return random.choice(best_actions)

    def update_q(self, state, action, reward, next_state, next_moves):
        old_q = self.get_q_value(state, action)

        if next_state is None:
            future_q = 0
        else:
            future_q = max([self.get_q_value(next_state, a) for a in next_moves], default=0)

        new_q = old_q + self.alpha * (reward + self.gamma * future_q - old_q)
        self.q_table[(state, action)] = new_q

    def decay_epsilon(self):
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay


In [5]:
# Agent = X (1)
# Opponent = Random O (-1)

In [6]:
def train_agent(episodes=20000):
    agent = QLearningAgent()
    game = TicTacToe()

    stats = {"win":0, "loss":0, "draw":0}

    for episode in range(episodes):
        game.reset()
        state = agent.get_state(game.board)

        while True:
            # AI move
            available = game.available_moves()
            action = agent.choose_action(state, available)
            game.make_move(action, 1)

            # Check win
            if game.current_winner == 1:
                agent.update_q(state, action, 1, None, [])
                stats["win"] += 1
                break

            if game.is_draw():
                agent.update_q(state, action, 0.5, None, [])
                stats["draw"] += 1
                break

            # Opponent (Random)
            opp_move = random.choice(game.available_moves())
            game.make_move(opp_move, -1)

            if game.current_winner == -1:
                agent.update_q(state, action, -1, None, [])
                stats["loss"] += 1
                break

            if game.is_draw():
                agent.update_q(state, action, 0.5, None, [])
                stats["draw"] += 1
                break

            next_state = agent.get_state(game.board)
            next_moves = game.available_moves()

            agent.update_q(state, action, 0, next_state, next_moves)

            state = next_state

        agent.decay_epsilon()

        if (episode+1) % 5000 == 0:
            print(f"Episode {episode+1}")

    return agent, stats


In [7]:
agent, stats = train_agent(30000)

print("Training Done")
print(stats)


Episode 5000
Episode 10000
Episode 15000
Episode 20000
Episode 25000
Episode 30000
Training Done
{'win': 27691, 'loss': 1297, 'draw': 1012}


In [8]:
def evaluate_agent(agent, games=1000):
    test_agent = agent
    test_agent.epsilon = 0  # Pure exploitation

    game = TicTacToe()
    results = {"win":0, "loss":0, "draw":0}

    for _ in range(games):
        game.reset()
        state = tuple(game.board)

        while True:
            # AI move
            available = game.available_moves()
            action = test_agent.choose_action(state, available)
            game.make_move(action, 1)

            if game.current_winner == 1:
                results["win"] += 1
                break

            if game.is_draw():
                results["draw"] += 1
                break

            # Random opponent
            opp_move = random.choice(game.available_moves())
            game.make_move(opp_move, -1)

            if game.current_winner == -1:
                results["loss"] += 1
                break

            if game.is_draw():
                results["draw"] += 1
                break

            state = tuple(game.board)

    return results


In [23]:
evaluate_agent(agent, 2000)


{'win': 1922, 'loss': 31, 'draw': 47}

In [10]:
import ipywidgets as widgets
from IPython.display import display


In [15]:
class TicTacToeGUI:
    def __init__(self, agent):
        self.agent = agent
        self.agent.epsilon = 0
        self.game = TicTacToe()

        self.buttons = []
        for i in range(9):
            button = widgets.Button(
                description="",
                layout=widgets.Layout(width='90px', height='90px'),
                style={'font_weight': 'bold'}
            )
            button.style.button_color = '#f0f0f0'
            button.on_click(self.make_click_handler(i))
            self.buttons.append(button)

        self.status = widgets.HTML(
            value="<h3 style='color:darkblue;'>🎮 Your Turn (X)</h3>"
        )

        self.reset_button = widgets.Button(
            description="🔄 Reset Game",
            layout=widgets.Layout(width='200px', height='40px')
        )
        self.reset_button.style.button_color = '#4CAF50'
        self.reset_button.on_click(self.reset_game)

        self.grid = widgets.GridBox(
            self.buttons,
            layout=widgets.Layout(
                grid_template_columns="repeat(3, 90px)",
                grid_gap="8px"
            )
        )

        display(self.status)
        display(self.grid)
        display(self.reset_button)

    def make_click_handler(self, position):
        def handler(b):
            if self.game.board[position] != 0 or self.game.current_winner:
                return

            # Human Move
            self.game.make_move(position, 1)
            self.update_buttons()

            if self.check_game_end():
                return

            # AI Move
            state = tuple(self.game.board)
            action = self.agent.choose_action(state, self.game.available_moves())
            self.game.make_move(action, -1)
            self.update_buttons()
            self.check_game_end()

        return handler

    def update_buttons(self):
        for i in range(9):
            value = self.game.board[i]

            if value == 1:
                self.buttons[i].description = "X"
                self.buttons[i].style.button_color = "#ADD8E6"
                self.buttons[i].style.font_weight = 'bold'

            elif value == -1:
                self.buttons[i].description = "O"
                self.buttons[i].style.button_color = "#FFB6C1"
                self.buttons[i].style.font_weight = 'bold'

            else:
                self.buttons[i].description = ""
                self.buttons[i].style.button_color = "#f0f0f0"

    def check_game_end(self):
        if self.game.current_winner == 1:
            self.status.value = "<h2 style='color:green;'>🎉 You Win!</h2>"
            self.highlight_winner(1)
            return True

        elif self.game.current_winner == -1:
            self.status.value = "<h2 style='color:red;'>🤖 AI Wins!</h2>"
            self.highlight_winner(-1)
            return True

        elif self.game.is_draw():
            self.status.value = "<h2 style='color:orange;'>🤝 It's a Draw!</h2>"
            return True

        else:
            self.status.value = "<h3 style='color:darkblue;'>🎮 Your Turn (X)</h3>"

        return False

    def highlight_winner(self, player):
        winning_lines = [
            [0,1,2],[3,4,5],[6,7,8],
            [0,3,6],[1,4,7],[2,5,8],
            [0,4,8],[2,4,6]
        ]

        for line in winning_lines:
            if all(self.game.board[i] == player for i in line):
                for i in line:
                    self.buttons[i].style.button_color = "#90EE90"

    def reset_game(self, b=None):
        self.game.reset()
        self.update_buttons()
        self.status.value = "<h3 style='color:darkblue;'>🎮 Your Turn (X)</h3>"


In [16]:
gui = TicTacToeGUI(agent)


HTML(value="<h3 style='color:darkblue;'>🎮 Your Turn (X)</h3>")

GridBox(children=(Button(layout=Layout(height='90px', width='90px'), style=ButtonStyle(button_color='#f0f0f0',…

Button(description='🔄 Reset Game', layout=Layout(height='40px', width='200px'), style=ButtonStyle(button_color…